# 1_Smart-Kage NOR postprocessing

This notebook turns the raw NOR data of **one cohort** into processed tables. Run it once per cohort; cohorts are
combined later in the analysis notebook.

**Inputs** (read only):
1. `<cohort>/DB/kageX/NOR_data.csv`: time of each wheel change and the objects shown
   (1 sample, 4 baseline, 2 test, 3 OiP; order 1, 4, 2, 3, 4, or 1, 4, 2, 4 without OiP)
2. `<cohort>/DB/kageX/analysis/drum_output/*.csv`: object exploration, one row = 0.5 s, `side` = left/right
3. `<cohort>/kage_descriptions.csv`: genotype, inclusion (`NOR_good`) and novel side in the test phase (`NOR_side`)

**What it does:**
- Defines each phase as wheel change → next wheel change (the last phase uses 24 h)
- Bins exploration into 1-min bins, keeping minutes without exploration as 0
- Total exploration per phase (first 24 h of each phase) and cumulative exploration
- Sample DI (baseline preference for the object and side that will be replaced by the novel object, first 24 h
  of the sample phase) and test DI (first 20 min, 2 h and 24 h)
- First approach (novel or familiar) and its latency in the test phase

**How to use:** set `COHORT_DIR` in the settings cell, then run all cells in order. Cohorts without OiP
need no changes; the phases are labelled from the object codes.

**Outputs** (`<cohort>/NOR_processed/`): 6 files, named `NOR_<what>_per_minute.csv` or `NOR_<what>_per_mouse.csv`
(full list at the end of the notebook). Every file has `KageID` and `Genotype`.

## 1. Settings

In [185]:
import numpy as np
import re
from pathlib import Path
from typing import List, Tuple, Optional
import pandas as pd
import sys

In [ ]:
# ------------------------------------------------------------------
# Settings: the only cell you need to edit
# ------------------------------------------------------------------

COHORT_DIR = Path('/Volumes/New Volume/2025-11-Nov-Dec_RELN-COLBOS_Coh1')

ROOT_DIR = COHORT_DIR / "DB"


OUTPUT_DIR = COHORT_DIR / "NOR_processed"     # per-cohort outputs
# OUTPUT_DIR = Path("/Users/jiatongsun/SmartKages_RELN-Colbos_analysis/2025-11-Nov-Dec_RELN-COLBOS_Coh1/NOR_processed") # temporary output dir for testing
OUTPUT_DIR.mkdir(exist_ok=True)

BIN_SIZE = 60        # 1-min bins
ROW_DURATION = 0.5   # one drum-output row = 0.5 s of exploration
PHASE_CAP_MIN = 24 * 60   # whole-phase measures (totals, sample DI, 24 h test DI) use at most the first 24 h of a phase


In [187]:
def extract_kage_id(path: Path) -> str:
    """
    Derive a normalized KageID (e.g., 'Kage1') from a folder path.
    - Looks for a pattern 'kage<digits>' (case-insensitive) in the folder name first.
    - If not found, searches upward through path parts.
    - Returns 'Kage_Unknown' if no match is found.

    Example:
    - 'kage1' -> 'Kage1'
    - 'KAGE 2' -> 'Kage2'
    """
    # Search the given folder's name for 'kage<number>'.
    m = re.search(r'kage\s*(\d+)', path.name, flags=re.IGNORECASE)
    if m:
        return f"Kage{m.group(1)}"
    # If not found in the immediate name, walk up the path parts in reverse to find a match.
    for part in path.parts[::-1]:
        m = re.search(r'kage\s*(\d+)', part, flags=re.IGNORECASE)
        if m:
            return f"Kage{m.group(1)}"
    # Fallback when the expected naming convention isn't present.
    return "Kage_Unknown"

def read_csv_with_fallbacks(csv_path: Path) -> Optional[pd.DataFrame]:
    """
    Robust CSV reader that:
    - Tries multiple common encodings: utf-8-sig, utf-8, cp1252, latin-1.
    - Uses 'sep=None' with the 'python' engine to infer delimiters (comma/semicolon/tab).
    - Skips malformed lines ('on_bad_lines="skip"') to avoid hard failures.

    Returns:
    - DataFrame if successfully read.
    - None if all attempts fail.
    """
    for enc in ("utf-8-sig", "utf-8", "cp1252", "latin-1"):
        try:
            return pd.read_csv(
                csv_path,
                encoding=enc,
                sep=None,              # Infer delimiter automatically.
                engine="python",       # Required for delimiter inference.
                on_bad_lines="skip",   # Prevent read errors due to bad rows.
            )
        except Exception:
            # Try the next encoding if this attempt fails.
            continue
    # All attempts failed; log a warning and return None.
    print(f"Warning: Failed to read {csv_path}")
    return None

def find_kage_nor_csvs(root: Path) -> List[Tuple[str, Path]]:
    r"""
    Discover all NOR_data.csv files directly under kage* directories within 'root'.

    Returns:
    - List of tuples: (KageID, csv_path)
      Where KageID is derived via 'extract_kage_id', and csv_path is root/kageX/NOR_data.csv.

    Scans structure like:
      <root>/kage1/NOR_data.csv
      <root>/kage2/NOR_data.csv
      ...
    """
    results: List[Tuple[str, Path]] = []
    # Matches folder names like 'kage1', 'KAGE 2', etc.
    kage_dir_regex = re.compile(r'kage\s*\d+', flags=re.IGNORECASE)

    # Validate the root before scanning.
    if not root.exists():
        print(f"Root directory does not exist: {root}")
        return results

    # Iterate through subdirectories whose names match the 'kage<number>' pattern.
    for kage_dir in sorted(p for p in root.iterdir() if p.is_dir() and kage_dir_regex.match(p.name)):
        kage_id = extract_kage_id(kage_dir)
        csv_path = kage_dir / "NOR_data.csv"
        # Record only when NOR_data.csv exists; otherwise notify.
        if csv_path.exists():
            results.append((kage_id, csv_path))
        else:
            print(f"Note: No NOR_data.csv in {kage_dir}")
    return results

def main():
    """
    Orchestrates the consolidation:
    - Finds all candidate NOR_data.csv files.
    - Loads each with encoding/delimiter inference.
    - Injects 'KageID' and 'SourceFile' at the front of each DataFrame.
    - Concatenates them and returns one table (one row per wheel change).
    """
    # Locate all kage*/NOR_data.csv pairs under ROOT_DIR.
    pairs = find_kage_nor_csvs(ROOT_DIR)
    if not pairs:
        print(f"No CSV files found under: {ROOT_DIR}")
        return
    print(f"Found {len(pairs)} NOR_data.csv files")

    frames: List[pd.DataFrame] = []
    total_files = 0

    # Load each CSV and annotate with its kage ID and source file path.
    for kage_id, csv_path in pairs:
        df = read_csv_with_fallbacks(csv_path)
        if df is None or df.empty:
            # Skip when unreadable or legitimately empty.
            print(f"Skipping empty/unreadable: {csv_path}")
            continue
        total_files += 1
        # Prepend metadata columns so they're visible first in the output.
        df.insert(0, "KageID", kage_id)              # e.g., 'Kage3'
        df.insert(1, "SourceFile", str(csv_path))    # Full absolute path to the source CSV
        frames.append(df)

    # Ensure we have data to combine.
    if not frames:
        print("No readable CSV data found after loading files.")
        return

    # Concatenate all loaded frames into a single DataFrame.
    combined = pd.concat(frames, ignore_index=True, sort=False)

    # Reorder columns to keep 'KageID' and 'SourceFile' at the front,
    # followed by the original data columns in their existing order.
    cols = list(combined.columns)
    for col in ("KageID", "SourceFile"):
        if col in cols:
            cols.remove(col)
    combined = combined[["KageID", "SourceFile"] + cols]

    print(f"Combined {total_files} files. Total rows: {len(combined):,}")
    return combined

# Combine all NOR_data.csv files
print(f"Python: {sys.executable}")
combined_nor = main()


def order_for_saving(df, first_cols=("KageID", "Genotype", "NOR_side", "Phase", "rotation")):
    """Order a table before saving.
    Rows: by kage number (Kage1, Kage2, ..., Kage10, not text order), then in time order (wheel change, then
    minute), so the phases of each mouse appear as they happened (pre_sample, sample, ITI baseline, test, ...).
    Columns: the mouse and phase columns in first_cols come first.
    """
    df = df.copy()
    df["_kage_number"] = df["KageID"].astype(str).str.extract(r"(\d+)", expand=False).astype(float)
    sort_cols = ["_kage_number"] + [c for c in ("rotation", "bin") if c in df.columns]
    df = df.sort_values(sort_cols, kind="stable").drop(columns="_kage_number").reset_index(drop=True)
    front = [c for c in first_cols if c in df.columns]
    return df[front + [c for c in df.columns if c not in front]]


Python: /opt/anaconda3/envs/smart-kages/bin/python
Found 9 NOR_data.csv files
Combined 9 files. Total rows: 45


## 2. Load the combined wheel-change table

The combined table has one row per wheel change.

Important columns:
- `KageID`: mouse/kage ID
- `rotation`: actual time the wheel changed
- `left_object` and `right_object`: object identity shown after the wheel changed

Object represents:
- `1` = training / sample
- `2` = test
- `3` = OiP
- `4` = baseline

Because the sequence is `1, 4, 2, 3, 4`,  (or `1, 4, 2, 4` without OiP) the two baseline periods need to be separated:
- first `4` = ITI baseline
- second `4` = post-trial baseline

For the main NOR analysis, we keep sample, ITI baseline and test, and exclude the post-trial baseline.

In [188]:
events = combined_nor.copy()

events["rotation"] = pd.to_datetime(events["rotation"])
events["set_time"] = pd.to_datetime(events["set_time"])

events.head()

,KageID,SourceFile,index,set_time,left_object,right_object,rotation
0,Kage1,/Volumes/New Volume/2025-11-Nov-Dec_RELN-COLBO...,0,2025-12-07 11:00:00,1,1,2025-12-07 11:19:29
1,Kage1,/Volumes/New Volume/2025-11-Nov-Dec_RELN-COLBO...,1,2025-12-08 11:00:00,4,4,2025-12-08 11:09:24
2,Kage1,/Volumes/New Volume/2025-11-Nov-Dec_RELN-COLBO...,2,2025-12-09 11:00:00,2,2,2025-12-09 11:09:26
3,Kage1,/Volumes/New Volume/2025-11-Nov-Dec_RELN-COLBO...,3,2025-12-10 12:00:00,3,3,2025-12-10 12:01:05
4,Kage1,/Volumes/New Volume/2025-11-Nov-Dec_RELN-COLBO...,4,2025-12-11 12:00:00,4,4,2025-12-11 12:02:20


In [189]:
# Label the phase of each wheel change
# Object codes:
# 1 = sample / training
# 2 = test
# 3 = object in place (OiP)
# 4 = baseline

# Since the sequence is 1, 4, 2, 3, 4, the first baseline is the ITI baseline
# and the second baseline is the post-trial baseline.

events = events.sort_values(["KageID", "rotation"]).copy()

events["object_code"] = events["left_object"]

# Count baseline occurrences within each Kage
# first 4 = ITI baseline, second 4 = post-trial baseline
events["baseline_number"] = (
    events["object_code"].eq(4)
    .groupby(events["KageID"])
    .cumsum()
)

events["Phase"] = np.select(
    [
        events["object_code"].eq(1),
        events["object_code"].eq(2),
        events["object_code"].eq(3),
        events["object_code"].eq(4) & events["baseline_number"].eq(1),
        events["object_code"].eq(4) & events["baseline_number"].ge(2),
    ],
    [
        "sample",
        "test",
        "OiP",
        "ITI_baseline",
        "post_trial_baseline",
    ],
    default="unknown"
)

events[["KageID", "set_time", "rotation", "left_object", "right_object", "baseline_number", "Phase"]].head(20)


,KageID,set_time,rotation,left_object,right_object,baseline_number,Phase
0,Kage1,2025-12-07 11:00:00,2025-12-07 11:19:29,1,1,0,sample
1,Kage1,2025-12-08 11:00:00,2025-12-08 11:09:24,4,4,1,ITI_baseline
2,Kage1,2025-12-09 11:00:00,2025-12-09 11:09:26,2,2,1,test
3,Kage1,2025-12-10 12:00:00,2025-12-10 12:01:05,3,3,1,OiP
4,Kage1,2025-12-11 12:00:00,2025-12-11 12:02:20,4,4,2,post_trial_baseline
5,Kage2,2025-12-07 11:00:00,2025-12-07 11:13:29,1,1,0,sample
6,Kage2,2025-12-08 11:00:00,2025-12-08 11:17:43,4,4,1,ITI_baseline
7,Kage2,2025-12-09 11:00:00,2025-12-09 11:12:34,2,2,1,test
8,Kage2,2025-12-10 12:00:00,2025-12-10 12:07:48,3,3,1,OiP
9,Kage2,2025-12-11 12:00:00,2025-12-11 12:23:00,4,4,2,post_trial_baseline


## Load kage_descriptions file

`kage_descriptions.csv` tells us:
- genotype
- whether the mouse should be included
- which side is novel during the test phase

`NOR_side` is the side of the novel object in the test phase (L or R; checked below). It is also used to
name the sides in the sample phase (sample DI) and for first approach.

In [190]:
DESC_CSV = COHORT_DIR / "kage_descriptions.csv" # change the directory and name of the file accordingly

desc = pd.read_csv(DESC_CSV)
desc.columns = desc.columns.str.strip()

desc["KageID"] = desc["Kage"].str.replace("kage", "Kage", regex=False)

desc.head()

,Kage,Group,Description,NOR_side,NOR_good,KageID
0,kage1,1,WT,L,yes,Kage1
1,kage2,1,WT,R,no,Kage2
2,kage3,2,RELN-Colbos,L,yes,Kage3
3,kage4,2,RELN-Colbos,R,yes,Kage4
4,kage5,1,WT,L,yes,Kage5


In [191]:
events = events.merge(
    desc[["KageID", "Description", "NOR_side", "NOR_good"]],
    on="KageID",
    how="left"
)

# events = events[events["NOR_good"] == "yes"].copy()
missing_desc = sorted(set(events.loc[events["Description"].isna(), "KageID"]))
if missing_desc:
    print("WARNING: in NOR_data but missing from kage_descriptions.csv (excluded):", missing_desc)

excluded = sorted(set(events.loc[events["NOR_good"].ne("yes") & events["Description"].notna(), "KageID"]))
if excluded:
    print("Excluded because NOR_good is not 'yes':", excluded)

events = events[events["NOR_good"] == "yes"].copy()
print(f"Mice kept: {events['KageID'].nunique()}")

# NOR_side: clean once here so every later step reads "L" or "R".
# Accepts L / R / Left / Right in any capitals; anything else (blank, typo) stops the notebook,
# because a wrong side would silently flip the mouse's DI, sample DI and first approach.
side_clean = (
    events["NOR_side"].astype(str).str.strip().str.upper()
    .map({"L": "L", "LEFT": "L", "R": "R", "RIGHT": "R"})
)
bad_side = events.loc[side_clean.isna(), ["KageID", "NOR_side"]].drop_duplicates()
if not bad_side.empty:
    raise ValueError(
        "NOR_side must be L or R in kage_descriptions.csv. Fix these kages and rerun:\n"
        + bad_side.to_string(index=False)
    )
events["NOR_side"] = side_clean


events[["KageID", "Phase", "Description", "NOR_side", "NOR_good"]].head()

Excluded because NOR_good is not 'yes': ['Kage2', 'Kage6', 'Kage7']
Mice kept: 6


,KageID,Phase,Description,NOR_side,NOR_good
0,Kage1,sample,WT,L,yes
1,Kage1,ITI_baseline,WT,L,yes
2,Kage1,test,WT,L,yes
3,Kage1,OiP,WT,L,yes
4,Kage1,post_trial_baseline,WT,L,yes


## 3. Define analysis windows between wheel changes

Each phase starts when the wheel rotates. The wheel only rotates while the mouse is inside a corridor drinking from the waterspout, so the animal cannot see the objects being changed. As a result, the actual wheel-change time (`rotation`) varies slightly from the scheduled time (`set_time`), but it mostly happens between 11:00 and 13:00.


We therefore use the exact time of wheel change. Instead of using a fixed 24-hour window, the analysis window is now:

`current wheel change → next wheel change`

This accounts for small timing differences caused by the mouse triggering the wheel change when it goes to drink.

The post-trial baseline is kept in the `events` table only so it can define the end of the test phase. It is then excluded from the main analysis table.

**Saved:** `NOR_wheel_changes.csv`: one row per wheel change (all phases, including the post-trial baseline),
with phase, genotype, `NOR_side`, start (`rotation`), end (`window_end`) and length (`window_hours`).


In [192]:
# Sort wheel changes chronologically for each mouse
events = events.sort_values(["KageID", "rotation"]).copy()

# Get the next wheel rotation for each mouse
events["next_rotation"] = (
    events.groupby("KageID")["rotation"]
    .shift(-1)
)

# Use next wheel change as the end of the analysis window
events["window_end"] = events["next_rotation"]

# For the last wheel change, fall back to 24 h
events["window_end"] = events["window_end"].fillna(
    events["rotation"] + pd.Timedelta(hours=24)
)
events["window_hours"] = (events["window_end"] - events["rotation"]).dt.total_seconds() / 3600


# For the main analysis, drop the post-trial baseline.
# It is not part of the main NOR comparison.
events_for_analysis = events[
    events["Phase"] != "post_trial_baseline"
].copy()


# one file with every wheel change; Description is saved as Genotype, as in the other files
order_for_saving(events.rename(columns={"Description": "Genotype"})).to_csv(OUTPUT_DIR / "NOR_wheel_changes.csv", index=False)

events[[
    "KageID",
    "Phase",
    "rotation",
    "next_rotation",
    "window_end",
    "window_hours"
]].head(20)


,KageID,Phase,rotation,next_rotation,window_end,window_hours
0,Kage1,sample,2025-12-07 11:19:29,2025-12-08 11:09:24,2025-12-08 11:09:24,23.831944
1,Kage1,ITI_baseline,2025-12-08 11:09:24,2025-12-09 11:09:26,2025-12-09 11:09:26,24.000556
2,Kage1,test,2025-12-09 11:09:26,2025-12-10 12:01:05,2025-12-10 12:01:05,24.860833
3,Kage1,OiP,2025-12-10 12:01:05,2025-12-11 12:02:20,2025-12-11 12:02:20,24.020833
4,Kage1,post_trial_baseline,2025-12-11 12:02:20,NaT,2025-12-12 12:02:20,24.000000
10,Kage3,sample,2025-12-07 11:01:06,2025-12-08 11:27:55,2025-12-08 11:27:55,24.446944
11,Kage3,ITI_baseline,2025-12-08 11:27:55,2025-12-09 11:16:26,2025-12-09 11:16:26,23.808611
12,Kage3,test,2025-12-09 11:16:26,2025-12-10 12:17:50,2025-12-10 12:17:50,25.023333
13,Kage3,OiP,2025-12-10 12:17:50,2025-12-11 12:04:17,2025-12-11 12:04:17,23.774167
14,Kage3,post_trial_baseline,2025-12-11 12:04:17,NaT,2025-12-12 12:04:17,24.000000


## Load drum_output files

The wheel-change table tells us:

- when a phase starts
- which phase it is

The drum output files tell us:

- when exploration occurred
- whether the mouse explored the left or right side

The goal is to extract exploration occurring after each wheel change.

In [193]:
drum_files = [
    f for f in sorted(ROOT_DIR.glob("kage*/analysis/drum_output/*.csv"))
    if not f.name.startswith("._")
]

print(f"Found {len(drum_files)} drum files")

Found 9 drum files


## Create a lookup table for drum files

This allows us to quickly find the correct drum file for each Kage.

In [194]:
drum_map = {
    f.parent.parent.parent.name.replace("kage", "Kage"): f
    for f in drum_files
}

list(drum_map.items())[:5]

[('Kage1',
  PosixPath('/Volumes/New Volume/2025-11-Nov-Dec_RELN-COLBOS_Coh1/DB/kage1/analysis/drum_output/kage1_drum.csv')),
 ('Kage2',
  PosixPath('/Volumes/New Volume/2025-11-Nov-Dec_RELN-COLBOS_Coh1/DB/kage2/analysis/drum_output/kage2_drum.csv')),
 ('Kage3',
  PosixPath('/Volumes/New Volume/2025-11-Nov-Dec_RELN-COLBOS_Coh1/DB/kage3/analysis/drum_output/kage3_drum.csv')),
 ('Kage4',
  PosixPath('/Volumes/New Volume/2025-11-Nov-Dec_RELN-COLBOS_Coh1/DB/kage4/analysis/drum_output/kage4_drum.csv')),
 ('Kage5',
  PosixPath('/Volumes/New Volume/2025-11-Nov-Dec_RELN-COLBOS_Coh1/DB/kage5/analysis/drum_output/kage5_drum.csv'))]

## 4. Extract exploration between wheel changes

For each wheel change, this extracts object exploration events from the matching drum file until the next wheel change.

This is used instead of a fixed 24-hour window because wheel-change timing can vary slightly depending on when the mouse goes to drink.

For the main analysis, the post-trial baseline is excluded, so the phases kept are:
- sample
- ITI_baseline
- test
- OiP

## 4.1 Build the binned exploration table between wheel changes (1 min bin)

For every included wheel change, this loop:
1. Loads the correct drum file.
2. Extracts exploration after the current wheel rotation and before the next wheel rotation.
3. Calculates time from wheel change.
4. Bins exploration into time windows.
5. Counts left and right exploration.
6. Converts counts into seconds.

- Creates a row for every 1-min bin of the window, with 0 s when there was no exploration, so later averages include all mice.
- Phases without any exploration are kept (all zeros) instead of being dropped.
- bin_duration_sec gives the length of each bin inside the window (the last bin can be shorter).
- left_frames / right_frames count the drum-output frames on each side in that minute (1 frame = 0.5 s);
  left_sec / right_sec / total_sec are the same in seconds.

In [195]:
# Extra window: the 2 h before the sample wheel change. No phase comes before sample, so without
# this the exploration before sample is never binned. It ends exactly at the sample wheel change.
PRE_SAMPLE_MIN = 120
pre_sample = events_for_analysis[events_for_analysis["Phase"].eq("sample")].copy()
pre_sample["window_end"] = pre_sample["rotation"]
pre_sample["rotation"] = pre_sample["rotation"] - pd.Timedelta(minutes=PRE_SAMPLE_MIN)
pre_sample["Phase"] = "pre_sample"
binning_events = pd.concat([pre_sample, events_for_analysis], ignore_index=True)


all_binned = []
drum_cache = {}   # load each drum file once (not once per phase)

for _, row in binning_events.iterrows():

    kage = row["KageID"]
    if kage not in drum_map:
        print("Missing drum file:", kage)
        continue

    if kage not in drum_cache:
        drum = pd.read_csv(drum_map[kage])
        drum["datetime"] = (
            pd.to_datetime(drum["date"].astype(str), format="%Y%m%d")
            + pd.to_timedelta(drum["timestamp"], unit="s")
        )
        drum_cache[kage] = drum
    drum = drum_cache[kage]

    start = row["rotation"]
    end = row["window_end"]

    # Exploration events inside this phase window
    session = drum[(drum["datetime"] >= start) & (drum["datetime"] < end)].copy()
    session["bin"] = ((session["datetime"] - start).dt.total_seconds() // BIN_SIZE).astype(int)

    # Every bin of the window, so bins without exploration are 0 (not missing)
    window_sec = (end - start).total_seconds()
    n_bins = int(np.ceil(window_sec / BIN_SIZE))

    if session.empty:
        print("No exploration (kept as zeros):", kage, row["Phase"], start)
        counts = pd.DataFrame(columns=["left", "right"])
    else:
        counts = pd.crosstab(session["bin"], session["side"])

    binned = (
        counts.reindex(index=range(n_bins), columns=["left", "right"], fill_value=0)
        .fillna(0)
        .astype(int)
        .rename_axis("bin")
        .reset_index()
    )
    binned.columns.name = None

    # Length of each bin inside the window (the last bin can be shorter than BIN_SIZE)
    binned["bin_duration_sec"] = np.minimum(BIN_SIZE, window_sec - binned["bin"] * BIN_SIZE)

    # left_frames / right_frames = number of drum-output frames on each side in that minute (1 frame = ROW_DURATION s)
    binned = binned.rename(columns={"left": "left_frames", "right": "right_frames"})
    binned["left_sec"] = binned["left_frames"] * ROW_DURATION
    binned["right_sec"] = binned["right_frames"] * ROW_DURATION
    binned["total_sec"] = binned["left_sec"] + binned["right_sec"]

    binned["KageID"] = kage
    binned["Genotype"] = row["Description"]
    binned["Phase"] = row["Phase"]
    binned["rotation"] = start
    binned["window_end"] = end

    # Clock time of each bin (bin n = rotation + n * BIN_SIZE up to the next bin or window_end)
    binned["bin_start"] = start + pd.to_timedelta(binned["bin"] * BIN_SIZE, unit="s")
    binned["bin_end"] = binned["bin_start"] + pd.to_timedelta(binned["bin_duration_sec"], unit="s")
    binned["NOR_side"] = row["NOR_side"]

    all_binned.append(binned)

binned_df = pd.concat(all_binned, ignore_index=True)
binned_df.head()


,bin,left_frames,right_frames,bin_duration_sec,left_sec,right_sec,total_sec,KageID,Genotype,Phase,rotation,window_end,bin_start,bin_end,NOR_side
0,0,0,0,60.0,0.0,0.0,0.0,Kage1,WT,pre_sample,2025-12-07 09:19:29,2025-12-07 11:19:29,2025-12-07 09:19:29,2025-12-07 09:20:29,L
1,1,0,0,60.0,0.0,0.0,0.0,Kage1,WT,pre_sample,2025-12-07 09:19:29,2025-12-07 11:19:29,2025-12-07 09:20:29,2025-12-07 09:21:29,L
2,2,0,0,60.0,0.0,0.0,0.0,Kage1,WT,pre_sample,2025-12-07 09:19:29,2025-12-07 11:19:29,2025-12-07 09:21:29,2025-12-07 09:22:29,L
3,3,0,0,60.0,0.0,0.0,0.0,Kage1,WT,pre_sample,2025-12-07 09:19:29,2025-12-07 11:19:29,2025-12-07 09:22:29,2025-12-07 09:23:29,L
4,4,0,0,60.0,0.0,0.0,0.0,Kage1,WT,pre_sample,2025-12-07 09:19:29,2025-12-07 11:19:29,2025-12-07 09:23:29,2025-12-07 09:24:29,L


## Total exploration per phase

One row per mouse per phase: left, right and total exploration (s). Phases can be a little longer than 24 h
(they end at the next wheel change), so only the first `PHASE_CAP_MIN` minutes (24 h) are summed; shorter phases
use their real length. `hours_used` gives the length that was summed.

Saved as `NOR_exploration_per_mouse.csv`.

In [196]:
capped = binned_df[binned_df["Phase"].ne("pre_sample") & binned_df["bin"].lt(PHASE_CAP_MIN)]

phase_total = (
    capped
    .groupby(["KageID", "Genotype", "NOR_side", "Phase", "rotation"], as_index=False)
    .agg(left_sec=("left_sec", "sum"), right_sec=("right_sec", "sum"),
         total_sec=("total_sec", "sum"), hours_used=("bin_duration_sec", "sum"))
)
phase_total["hours_used"] = (phase_total["hours_used"] / 3600).round(3)

phase_total = order_for_saving(phase_total)
phase_total.to_csv(OUTPUT_DIR / "NOR_exploration_per_mouse.csv", index=False)
phase_total.head()

,KageID,Genotype,NOR_side,Phase,rotation,left_sec,right_sec,total_sec,hours_used
0,Kage1,WT,L,sample,2025-12-07 11:19:29,219.5,493.0,712.5,23.832
1,Kage1,WT,L,ITI_baseline,2025-12-08 11:09:24,57.0,116.0,173.0,24.000
2,Kage1,WT,L,test,2025-12-09 11:09:26,110.5,135.5,246.0,24.000
3,Kage1,WT,L,OiP,2025-12-10 12:01:05,352.0,349.5,701.5,24.000
4,Kage3,RELN-Colbos,L,sample,2025-12-07 11:01:06,369.0,515.0,884.0,24.000


## 4.2 Cumulative exploration over time

This shows how total exploration builds up after each wheel change.

For each mouse and phase, we cumulatively sum exploration across time bins.

Because every 1-min bin exists, cumulative values are defined for every minute (flat while the mouse is not exploring). Saved as `NOR_exploration_per_minute.csv` (all phases, whole phase, plus the 2 h `pre_sample` window), which the
analysis notebook combines into `NOR_MASTER_binned.csv`.

In [197]:
# Kage number, then time order (pre_sample, sample, ITI baseline, test, ...), then minute; mouse and phase columns first
binned_df = order_for_saving(
    binned_df,
    first_cols=("KageID", "Genotype", "NOR_side", "Phase", "rotation", "window_end",
                "bin", "bin_start", "bin_end", "bin_duration_sec"),
)

binned_df["cumulative_left_sec"] = (
    binned_df.groupby(["KageID", "Phase", "rotation"])["left_sec"].cumsum()
)

binned_df["cumulative_right_sec"] = (
    binned_df.groupby(["KageID", "Phase", "rotation"])["right_sec"].cumsum()
)

binned_df["cumulative_total_sec"] = (
    binned_df.groupby(["KageID", "Phase", "rotation"])["total_sec"].cumsum()
)

binned_df.head()

,KageID,Genotype,NOR_side,Phase,rotation,window_end,bin,bin_start,bin_end,bin_duration_sec,left_frames,right_frames,left_sec,right_sec,total_sec,cumulative_left_sec,cumulative_right_sec,cumulative_total_sec
0,Kage1,WT,L,pre_sample,2025-12-07 09:19:29,2025-12-07 11:19:29,0,2025-12-07 09:19:29,2025-12-07 09:20:29,60.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
1,Kage1,WT,L,pre_sample,2025-12-07 09:19:29,2025-12-07 11:19:29,1,2025-12-07 09:20:29,2025-12-07 09:21:29,60.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,Kage1,WT,L,pre_sample,2025-12-07 09:19:29,2025-12-07 11:19:29,2,2025-12-07 09:21:29,2025-12-07 09:22:29,60.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,Kage1,WT,L,pre_sample,2025-12-07 09:19:29,2025-12-07 11:19:29,3,2025-12-07 09:22:29,2025-12-07 09:23:29,60.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
4,Kage1,WT,L,pre_sample,2025-12-07 09:19:29,2025-12-07 11:19:29,4,2025-12-07 09:23:29,2025-12-07 09:24:29,60.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [198]:
# save the cumulative exploration file
binned_df.to_csv(OUTPUT_DIR / "NOR_exploration_per_minute.csv", index=False)

## Convert left/right exploration into novel/familiar exploration

The drum file records left and right exploration.

For test sessions we convert these into exploration of the novel object and familiar object using the NOR_side information from kage_descriptions.

In [199]:
test_df = binned_df[
    binned_df["Phase"] == "test"
].copy()

In [200]:
test_df["NOR_side"].value_counts()

NOR_side
L    5924
R    3018
Name: count, dtype: int64

In [201]:
test_df["novel_sec"] = np.where(
    test_df["NOR_side"] == "L",
    test_df["left_sec"],
    test_df["right_sec"]
)

test_df["familiar_sec"] = np.where(
    test_df["NOR_side"] == "L",
    test_df["right_sec"],
    test_df["left_sec"]
)

## Sample phase DI (baseline preference)

In the sample phase the mouse explores two different objects, both new to it (object A and object B). In the test
phase the object on the `NOR_side` side (A) is replaced by a novel object (C), and the other (B) stays as the
familiar object (B'). Each side is named by what it will hold in the test phase: the "novel side" is where A is
now and C will appear.

sample DI = (novel side − familiar side) / (novel side + familiar side) = (A − B) / (A + B)

- Uses the first `PHASE_CAP_MIN` minutes (24 h) of the sample phase; a shorter phase uses its real length.
- It is the mouse's preference for the object and side that will be replaced, before anything is novel:
  side preference plus any preference for object A over object B.
- Around 0 = no baseline preference. A positive value means the mouse already preferred the future novel side
  (or object A), so part of its test DI could come from that rather than memory; ΔDI (test − sample) removes it.
- NaN if the mouse did not explore during the sample phase.

Added to both DI files below (`sample_DI`).

In [202]:
sample = binned_df[binned_df["Phase"].eq("sample") & binned_df["bin"].lt(PHASE_CAP_MIN)].copy()
novel_is_left = sample["NOR_side"].eq("L")
sample["novel_side_sec"] = np.where(novel_is_left, sample["left_sec"], sample["right_sec"])
sample["familiar_side_sec"] = np.where(novel_is_left, sample["right_sec"], sample["left_sec"])

sample_di = (
    sample.groupby("KageID", as_index=False)
    .agg(sample_novel_side_sec=("novel_side_sec", "sum"),
         sample_familiar_side_sec=("familiar_side_sec", "sum"),
         sample_hours_used=("bin_duration_sec", "sum"))
)
sample_di["sample_hours_used"] = (sample_di["sample_hours_used"] / 3600).round(3)
total = sample_di["sample_novel_side_sec"] + sample_di["sample_familiar_side_sec"]
sample_di["sample_DI"] = ((sample_di["sample_novel_side_sec"] - sample_di["sample_familiar_side_sec"]) / total).where(total > 0)

sample_di.head()

,KageID,sample_novel_side_sec,sample_familiar_side_sec,sample_hours_used,sample_DI
0,Kage1,219.5,493.0,23.832,-0.383860
1,Kage3,369.0,515.0,24.000,-0.165158
2,Kage4,320.5,514.0,24.000,-0.231875
3,Kage5,396.0,410.5,24.000,-0.017979
4,Kage8,612.0,456.0,23.109,0.146067


## 5. Novelty/Familiar Discrimination Index (DI)

DI measures preference for the novel object.

DI = (novel − familiar) / (novel + familiar)

where novel and familiar are the time (s) spent exploring each object.

- DI > 0 = novel preference
- DI = 0 = no preference
- DI < 0 = familiar preference

Here we calculate a **cumulative DI**: at each minute, novel and familiar exploration are summed from the
wheel change up to and including that minute, and DI is calculated from these totals. We don't use a
per-minute DI, because in a single minute a mouse usually explores only one object (DI = ±1) or none
(DI undefined), so per-minute values are mostly noise.

DI is NaN until the mouse has explored at least one object.

Saved as `NOR_DI_per_minute.csv` (test phase, one row per mouse per minute: novel_sec, familiar_sec,
cumulative_novel_sec, cumulative_familiar_sec, cumulative_DI). Each row also has the mouse's `sample_DI`, so the
cumulative test DI can be compared with it at any minute.


In [203]:
# cumulative DI: novel and familiar time summed from the wheel change up to and including each minute
test_df = test_df.sort_values(["KageID", "rotation", "bin"])
test_df["cumulative_novel_sec"] = test_df.groupby(["KageID", "rotation"])["novel_sec"].cumsum()
test_df["cumulative_familiar_sec"] = test_df.groupby(["KageID", "rotation"])["familiar_sec"].cumsum()

cumulative_total = test_df["cumulative_novel_sec"] + test_df["cumulative_familiar_sec"]
test_df["cumulative_DI"] = (
    (test_df["cumulative_novel_sec"] - test_df["cumulative_familiar_sec"]) / cumulative_total
).where(cumulative_total > 0)

di_table = test_df[
    ["KageID", "Genotype", "NOR_side", "Phase", "rotation", "bin",
     "novel_sec", "familiar_sec",
     "cumulative_novel_sec", "cumulative_familiar_sec", "cumulative_DI"]
].merge(sample_di[["KageID", "sample_DI"]], on="KageID", how="left")

di_table = order_for_saving(di_table)
di_table.to_csv(OUTPUT_DIR / "NOR_DI_per_minute.csv", index=False)



## DI per mouse: sample vs test (20 min, 2 h, 24 h)

One row per mouse, so sample and test DI can be compared directly.

- **Sample:** first 24 h of the sample phase (see above).
- **Test:** for each window in `TEST_WINDOWS_MIN`, novel and familiar time are summed from the test wheel change
  to the end of the window. 20 min is the primary window, 2 h secondary, 24 h a sensitivity check.
- `delta_DI_<window>` = test DI − sample DI.
- DI is NaN if the mouse explored neither object in that window.

Saved as `NOR_DI_per_mouse.csv`.

In [204]:
TEST_WINDOWS_MIN = {"20min": 20, "2h": 120, "24h": PHASE_CAP_MIN}   # name in the column -> minutes after the test wheel change

di_per_mouse = (
    events_for_analysis.loc[events_for_analysis["Phase"].eq("test"), ["KageID", "Description", "NOR_side"]]
    .rename(columns={"Description": "Genotype"})
    .merge(sample_di, on="KageID", how="left")
)

for name, cutoff in TEST_WINDOWS_MIN.items():
    window = (
        test_df[test_df["bin"] < cutoff]
        .groupby("KageID", as_index=False)
        .agg(**{f"test_novel_sec_{name}": ("novel_sec", "sum"),
                f"test_familiar_sec_{name}": ("familiar_sec", "sum")})
    )
    novel, familiar = window[f"test_novel_sec_{name}"], window[f"test_familiar_sec_{name}"]
    window[f"test_DI_{name}"] = ((novel - familiar) / (novel + familiar)).where(novel + familiar > 0)
    di_per_mouse = di_per_mouse.merge(window, on="KageID", how="left")
    di_per_mouse[f"delta_DI_{name}"] = di_per_mouse[f"test_DI_{name}"] - di_per_mouse["sample_DI"]

di_per_mouse = order_for_saving(di_per_mouse)
di_per_mouse.to_csv(OUTPUT_DIR / "NOR_DI_per_mouse.csv", index=False)
di_per_mouse[["KageID", "Genotype", "sample_DI"] + [f"test_DI_{n}" for n in TEST_WINDOWS_MIN]]

,KageID,Genotype,sample_DI,test_DI_20min,test_DI_2h,test_DI_24h
0,Kage1,WT,-0.383860,0.175676,0.147368,-0.101626
1,Kage3,RELN-Colbos,-0.165158,0.015625,0.449057,0.251422
2,Kage4,RELN-Colbos,-0.231875,0.377049,-0.226994,-0.077869
3,Kage5,WT,-0.017979,0.228346,-0.111111,0.035179
4,Kage8,RELN-Colbos,0.146067,0.147475,0.152137,0.065533
5,Kage9,WT,-0.148560,0.215190,0.264317,0.258803


# 6. First approach analysis

First approach asks: **after the test-phase wheel change, which object does the mouse explore first?**
Mice that remember the familiar object are expected to go to the novel one first.

For each mouse:
1. Take the drum data from the test-phase wheel rotation to 2 hours (could be changed by modifying the `FIRST_APPROACH_WINDOW_MIN`) after it.
2. Find the first drum row in that window, which is the first 0.5-s frame of object exploration.
3. Record its side (left or right) and compare it with `NOR_side` from kage_descriptions
   (the side of the novel object): same side = **novel**, other side = **familiar**.

This gives one value per mouse. It doesn't depend on how long the mouse explores, so it complements the DI.
It is based on a single event, so it is best summarised as the proportion of mice that approach the novel
object first (per genotype), not as a per-mouse score.

Mice with no exploration in the window are kept with FirstApproach = "none" (and no latency).

Saved as `NOR_first_approach_per_mouse.csv` (one row per mouse: KageID, Genotype, NOR_side, FirstSide,
FirstApproach, Latency_sec = seconds from the wheel change to the first exploration).


In [205]:
FIRST_APPROACH_WINDOW_MIN = 120   # look for the first exploration within this many minutes after the test wheel change

first_approach_results = []

for _, row in events_for_analysis[events_for_analysis["Phase"] == "test"].iterrows():

    kage = row["KageID"]
    drum = drum_cache[kage]   # loaded in the binning loop above

    start = row["rotation"]
    end = min(start + pd.Timedelta(minutes=FIRST_APPROACH_WINDOW_MIN), row["window_end"])

    session = drum[(drum["datetime"] >= start) & (drum["datetime"] < end)].sort_values("datetime")

    if session.empty:
        # no exploration in the window: keep the mouse so group sizes stay correct
        first_side, first_approach, latency_sec = None, "none", np.nan
    else:
        first = session.iloc[0]
        first_side = first["side"]
        novel_side = "left" if row["NOR_side"] == "L" else "right"
        first_approach = "novel" if first_side == novel_side else "familiar"
        latency_sec = (first["datetime"] - start).total_seconds()

    first_approach_results.append({
        "KageID": kage,
        "Genotype": row["Description"],
        "NOR_side": row["NOR_side"],
        "FirstSide": first_side,
        "FirstApproach": first_approach,
        "Latency_sec": latency_sec,
    })


In [206]:
first_approach_df = pd.DataFrame(
    first_approach_results
)

first_approach_df.head()

,KageID,Genotype,NOR_side,FirstSide,FirstApproach,Latency_sec
0,Kage1,WT,L,right,familiar,110.0
1,Kage3,RELN-Colbos,L,left,novel,25.0
2,Kage4,RELN-Colbos,R,right,novel,13.0
3,Kage5,WT,L,left,novel,446.5
4,Kage8,RELN-Colbos,L,left,novel,18.0


In [207]:
first_approach_df = order_for_saving(first_approach_df)
first_approach_df.to_csv(
    OUTPUT_DIR / "NOR_first_approach_per_mouse.csv",
    index=False
)

### Before proceeding, check your output files

`<cohort>/NOR_processed/` should contain these 6 files:

| File | One row per | Contents |
|---|---|---|
| `NOR_wheel_changes.csv` | wheel change | phase, genotype, `NOR_side`, start, end and length of each phase |
| `NOR_exploration_per_minute.csv` | mouse × phase × minute | left / right / total exploration (s) and cumulative; whole phase + 2 h `pre_sample` (main table) |
| `NOR_exploration_per_mouse.csv` | mouse × phase | left / right / total exploration (s) in the first 24 h of the phase |
| `NOR_DI_per_minute.csv` | mouse × test minute | novel / familiar (s), cumulative DI, and `sample_DI` |
| `NOR_DI_per_mouse.csv` | mouse | `sample_DI`, `test_DI_20min` / `_2h` / `_24h` and `delta_DI_*` |
| `NOR_first_approach_per_mouse.csv` | mouse | first object explored in the test (novel / familiar / none) and latency (s) |

In every file the rows are in kage-number order (Kage1, Kage2, …, Kage10), then in time order, so each mouse's
phases appear as they happened (pre_sample, sample, ITI baseline, test, OiP, post-trial baseline); the mouse and
phase columns (`KageID`, `Genotype`, `NOR_side`, `Phase`, `rotation`) come first.

Files from older versions of this notebook (e.g. `NOR_24h_*`, `NOR_test_*`, `NOR_events_*`) are no longer
written or read; they can be deleted.

### Run the notebook again for all your cohorts and move to `2_SmartKage_NOR_analysis.ipynb`